# UDA-Hub chat notebook

> **For reviewers**  
> This notebook preserves Udacity's starter chat example as a reference. The submitted UDA-Hub application is [03_agentic_app.py](03_agentic_app.py), which runs the LangGraph workflow in [agentic/workflow.py](agentic/workflow.py). The runnable cells below call that same application. See the [README demo](README.md#run-the-support-demo) for terminal commands.

## Prepare the demo

Open this notebook from the repository root and select the project's Python environment. Set up the synthetic databases with [01_external_db_setup.ipynb](01_external_db_setup.ipynb) and [02_core_db_setup.ipynb](02_core_db_setup.ipynb) if you haven't already. The examples run locally without an API key. Each example starts with a fresh demo ticket and writes its normal local records and logs.

In [1]:
import json
from contextlib import closing
from pathlib import Path
import sqlite3
import subprocess
import sys
from uuid import uuid4

root = Path.cwd()
app = root / "03_agentic_app.py"
cultpass_db = root / "data/external/cultpass.db"
core_db = root / "data/core/udahub.db"

if not app.is_file():
    raise FileNotFoundError("Set the notebook working directory to the repository root.")
if not (cultpass_db.is_file() and core_db.is_file()):
    raise FileNotFoundError("Run setup notebooks 01 and 02 before running this demo.")

def run_app(*arguments):
    command = [sys.executable, "-B", str(app), *arguments]
    result = subprocess.run(command, cwd=root, capture_output=True, text=True, timeout=45)
    if result.returncode:
        raise RuntimeError(result.stdout.strip() or result.stderr.strip())
    print(json.dumps(json.loads(result.stdout), indent=2))


def run_ticket(text, *options, ticket_id=None, turn_id="first"):
    ticket_id = ticket_id or f"notebook-{uuid4().hex}"
    run_app("--ticket-id", ticket_id, "--turn-id", turn_id, *options, "--", text)

## Resolve a sign-in ticket

The answer cites the CultPass sign-in article. Edit the text to try another support request.

In [2]:
run_ticket("I forgot my password. How can I sign in again?")

{
  "decision": {
    "next_step": "resolve",
    "reason": "evidence_ready"
  },
  "response": {
    "text": "Try tapping 'Forgot Password' on the login screen. Make sure you're using the email associated with your account. If the email doesn't arrive, check spam or try again in a few minutes.",
    "citations": [
      {
        "article_id": "33b6098e-cf40-460c-8307-1e92781ce106",
        "intent_id": "login",
        "title": "How to Handle Login Issues?",
        "excerpt": "Try tapping 'Forgot Password' on the login screen. Make sure you're using the email associated with your account. If the email doesn't arrive, check spam or try again in a few minutes."
      }
    ]
  },
  "trace_status": "recorded",
  "memory_status": "skipped"
}


## Look up a subscription

Use a synthetic profile to get a personalized answer from CultPass account and subscription data.

In [3]:
with closing(sqlite3.connect(f"{cultpass_db.as_uri()}?mode=ro", uri=True)) as db:
    profile = db.execute(
        "SELECT u.user_id FROM users u JOIN subscriptions s USING (user_id) "
        "WHERE u.is_blocked = 0 ORDER BY u.user_id LIMIT 1"
    ).fetchone()
if profile is None:
    raise RuntimeError("The CultPass demo database needs an active customer with a subscription.")

run_ticket(
    "What is my subscription status and monthly quota?",
    "--demo-customer-id", profile[0],
)

{
  "decision": {
    "next_step": "resolve",
    "reason": "evidence_ready"
  },
  "response": {
    "text": "Your CultPass subscription is active on the premium plan. Your monthly quota is 5 experiences. Open My Account and select Manage Plan to review your subscription details.",
    "citations": [
      {
        "article_id": "a7d44199-5ae8-4f4d-8caa-dccb6b846309",
        "intent_id": "personal_subscription_status",
        "title": "Check Subscription Status and Monthly Quota",
        "excerpt": "Open My Account and select Manage Plan to review your subscription details."
      }
    ]
  },
  "trace_status": "recorded",
  "memory_status": "saved",
  "thread_id": "c32318c8355cc9903df167fed0cfb843d0d93a55363565f64e19dbe6be7f092f"
}


## Escalate a question outside the knowledge base

The workflow returns a handoff with an insufficient-evidence reason.

In [4]:
run_ticket("How do I assemble a telescope?")

{
  "decision": {
    "next_step": "escalate",
    "reason": "insufficient_evidence"
  },
  "handoff": {
    "reason": "insufficient_evidence",
    "summary": "Please review this ticket about unclassified request.",
    "actions": [
      "classified",
      "knowledge:insufficient_evidence"
    ]
  },
  "trace_status": "recorded",
  "memory_status": "skipped"
}


## Continue a reservation conversation

Run these cells in order after the subscription example. Both turns use the same synthetic profile and ticket. The follow-up only asks for the next step. A booking citation shows that the workflow recalled the context. The final cell inspects the latest checkpoint and shows the second turn's classification, evidence, and decision. Rerun the first cell to start a fresh conversation.

In [5]:
booking_ticket = f"notebook-booking-{uuid4().hex}"
booking_profile = ("--demo-customer-id", profile[0])

run_ticket(
    "How do I reserve a spot for an event?",
    *booking_profile,
    ticket_id=booking_ticket,
    turn_id="first",
)

{
  "decision": {
    "next_step": "resolve",
    "reason": "evidence_ready"
  },
  "response": {
    "text": "You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping 'Reserve'. Be sure to arrive 15 minutes early with your QR code ready.",
    "citations": [
      {
        "article_id": "680d2fdb-bbc6-41f0-8a2d-fc82b502a71d",
        "intent_id": "booking",
        "title": "How to Reserve a Spot for an Event",
        "excerpt": "You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping 'Reserve'. Be sure to arrive 15 minutes early with your QR code ready."
      }
    ]
  },
  "trace_status": "recorded",
  "memory_status": "saved",
  "thread_id": "f639b9d3ea015661e450e356849ee76b336ac5aa496af3fad014a7a61f3118bf"
}


In [6]:
run_ticket(
    "What should I do next?",
    *booking_profile,
    ticket_id=booking_ticket,
    turn_id="second",
)

{
  "decision": {
    "next_step": "resolve",
    "reason": "evidence_ready"
  },
  "response": {
    "text": "You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping 'Reserve'. Be sure to arrive 15 minutes early with your QR code ready.",
    "citations": [
      {
        "article_id": "680d2fdb-bbc6-41f0-8a2d-fc82b502a71d",
        "intent_id": "booking",
        "title": "How to Reserve a Spot for an Event",
        "excerpt": "You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping 'Reserve'. Be sure to arrive 15 minutes early with your QR code ready."
      }
    ]
  },
  "trace_status": "recorded",
  "memory_status": "saved",
  "thread_id": "f639b9d3ea015661e450e356849ee76b336ac5aa496af3fad014a7a61f3118bf"
}


In [7]:
run_app("--ticket-id", booking_ticket, *booking_profile, "--inspect")

{
  "thread_id": "f639b9d3ea015661e450e356849ee76b336ac5aa496af3fad014a7a61f3118bf",
  "checkpoint": {
    "turn_id": "second",
    "text": "What should I do next?",
    "classification": {
      "intents": [
        {
          "intent_id": "booking",
          "category": "reservations",
          "terms": [
            "event",
            "reservation"
          ],
          "customer_data_required": false
        }
      ],
      "human_support_requested": false,
      "sensitive_action_requested": false,
      "customer_data_required": false
    },
    "decision": {
      "next_step": "resolve",
      "reason": "evidence_ready"
    },
    "retrieval": {
      "status": "ready",
      "intents": [
        {
          "intent_id": "booking",
          "candidates": [
            {
              "article_id": "680d2fdb-bbc6-41f0-8a2d-fc82b502a71d",
              "title": "How to Reserve a Spot for an Event",
              "content": "If a user asks how to reserve an event:\n\n- Guid

## Udacity starter chat (reference)

The original interactive example uses the [sample orchestrator](agentic/starter_workflow.py), a model credential, and a fixed thread. It is shown here as a code snippet so **Run All** can complete without interactive input.

```python
from dotenv import load_dotenv
from utils import chat_interface
load_dotenv()
from agentic.starter_workflow import orchestrator

chat_interface(orchestrator, "1")
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]
```